# 🎨 Moebius 0.22B Manga Inpainting & Decensoring (Local Data Mode)
This notebook runs **hustvl/Moebius** on a free Google Colab **T4 GPU** using pre-saved local images and masks.

### 1. Check GPU
Ensure you are on a GPU runtime (**Runtime -> Change runtime type -> T4 GPU**).

In [ ]:
!nvidia-smi

### 2. Clone Repository & Install Dependencies

In [ ]:
%cd /content
!git clone https://github.com/hustvl/Moebius.git
%cd /content/Moebius

!pip install -q diffusers transformers accelerate gradio pillow torchvision huggingface_hub einops timm matplotlib

### 3. Download Moebius Weights

In [ ]:
import os
from huggingface_hub import hf_hub_download

os.makedirs("/content/Moebius/weight/Moebius/pretrained", exist_ok=True)
os.makedirs("/content/Moebius/weight/vae", exist_ok=True)

print("Downloading Moebius weights from Hugging Face...")
hf_hub_download(
    repo_id="hustvl/Moebius",
    filename="pretrained/diffusion_pytorch_model.bin",
    local_dir="/content/Moebius/weight/Moebius",
    local_dir_use_symlinks=False
)

hf_hub_download(
    repo_id="madebyollin/sdxl-vae-fp16-fix",
    filename="diffusion_pytorch_model.bin",
    local_dir="/content/Moebius/weight/vae",
    local_dir_use_symlinks=False
)
hf_hub_download(
    repo_id="madebyollin/sdxl-vae-fp16-fix",
    filename="config.json",
    local_dir="/content/Moebius/weight/vae",
    local_dir_use_symlinks=False
)
print("✅ Weights ready!")

### 4. Load Moebius Pipeline into GPU Memory

In [ ]:
import sys
import time
import glob
import random
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from types import SimpleNamespace

if "/content/Moebius" not in sys.path:
    sys.path.insert(0, "/content/Moebius")

from utils_infer import build_pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"

args = SimpleNamespace(
    model_config="config/model_cfg/moebius.yaml",
    model_weight="weight/Moebius/pretrained/diffusion_pytorch_model.bin",
    device=device,
)

print(f"Loading Moebius pipeline onto {device}...")
pipeline = build_pipeline(args)
print("🚀 Model loaded and ready in GPU memory!")

def pad_to_square(image: Image.Image, mask: Image.Image):
    w, h = image.size
    max_dim = max(w, h)
    pad_left = (max_dim - w) // 2
    pad_top = (max_dim - h) // 2

    square_img = Image.new("RGB", (max_dim, max_dim), (255, 255, 255))
    square_img.paste(image, (pad_left, pad_top))

    square_mask = Image.new("L", (max_dim, max_dim), 0)
    square_mask.paste(mask, (pad_left, pad_top))

    box = (pad_left, pad_top, pad_left + w, pad_top + h)
    return square_img, square_mask, box

def inpaint_image_mask(image: Image.Image, mask: Image.Image, cfg=2.5, dilation=8, num_steps=20, seed=42):
    sq_img, sq_mask, crop_box = pad_to_square(image, mask)
    seed_val = int(seed) if seed >= 0 else random.randint(1, 2147483647)

    t0 = time.perf_counter()
    with torch.inference_mode():
        results = pipeline(
            [sq_img],
            [sq_mask],
            guidance_scale=float(cfg),
            mask_dilate_kernel_size=int(dilation),
            num_steps=int(num_steps),
            retry=seed_val,
            paste=True,
            compensate=False,
            noise_offset=0.0357,
        )
    infer_time = round(time.perf_counter() - t0, 2)

    result_sq_img = results[0]
    result_sq_full = result_sq_img.resize(sq_img.size, Image.Resampling.LANCZOS)
    result_img = result_sq_full.crop(crop_box).resize(image.size, Image.Resampling.LANCZOS)
    return result_img, infer_time

### 5. Run Inpainting on Local Saved Data (`./input` or `./devscripts/input`)
Upload your `devscripts/input` folder to `/content/input` (or run locally in the repo) to batch inpaint and view results inline.

In [ ]:
input_dir = Path("./devscripts/input")
if not input_dir.exists():
    input_dir = Path("/content/input")
    input_dir.mkdir(parents=True, exist_ok=True)

output_dir = Path("./outputs/colab_test")
output_dir.mkdir(parents=True, exist_ok=True)

image_files = sorted(list(input_dir.glob("*_image.png")) + list(input_dir.glob("*_image.jpg")))
print(f"Found {len(image_files)} sample(s) in {input_dir}:")

for img_path in image_files:
    prefix = img_path.name.replace("_image.png", "").replace("_image.jpg", "")
    mask_candidates = list(input_dir.glob(f"{prefix}_mask.*"))
    if not mask_candidates:
        print(f"Skipping {img_path.name} (no matching mask found)")
        continue
    
    mask_path = mask_candidates[0]
    print(f"\nProcessing: {prefix} ...")
    orig_img = Image.open(img_path).convert("RGB")
    mask_img = Image.open(mask_path).convert("L")
    
    result, elapsed = inpaint_image_mask(orig_img, mask_img, cfg=2.5, dilation=8, num_steps=20)
    out_file = output_dir / f"{prefix}_inpainted.png"
    result.save(out_file)
    print(f"✨ Saved -> {out_file} (Inferred in {elapsed}s)")

    # Display side-by-side comparison in notebook
    fig, axes = plt.subplots(1, 3, figsize=(15, 6))
    axes[0].imshow(orig_img)
    axes[0].set_title("Original Image")
    axes[0].axis("off")
    
    axes[1].imshow(mask_img, cmap="gray")
    axes[1].set_title("Mask")
    axes[1].axis("off")
    
    axes[2].imshow(result)
    axes[2].set_title(f"Inpainted ({elapsed}s)")
    axes[2].axis("off")
    plt.tight_layout()
    plt.show()